# Workflow 11

## Configuration

In [1]:
prodowndir = "chemfiles/proteindownload/"
ligdowndir = "chemfiles/liganddownload/"
prodir = "chemfiles/protein/"
ligdir = "chemfiles/ligand/"
resultsdir = "results/"
docked = resultsdir + "docked.sdf"
log = resultsdir + "gninalog.txt"
rmsdlog = resultsdir + "rmsd.txt"
masterlog = resultsdir + "masterlog.txt"

In [2]:
experimentdescription = "Workflow 11 is Andy's 'Top 9' experiment extended. We will report results for top 9 poses but with multiple PDBFixer calls."

In [3]:
proteindict = {}
liganddict = {}
gninaoptdict = {}
prepdict = {}

In [4]:
import copy
def reportdict(rows, columns):
    lines = []
    if len(rows) == 0:
        return (lines) 
    inner_keys = set()
    for r in rows.values():
        inner_keys.update(r.keys())
    col_widths = {}
    col_widths["id"] = max(len("id"), max(len(k) for k in rows))
    for col in inner_keys:
        header_len = len(col)
        data_len = max(len(str(r.get(col, ""))) for r in rows.values())
        col_widths[col] = max(header_len, data_len)
    header = "  ".join(f"{col:<{col_widths[col]}}" for col in columns)
    lines.append(header)
    for outer_key, inner in rows.items():
        cells = [f"{outer_key:<{col_widths['id']}}"]
        for col in columns[1:]:
            cells.append(f"{str(inner.get(col, '')):<{col_widths[col]}}")
        lines.append("  ".join(cells))
    return (lines)

## Download Ideal Ligand files from Pubchem

In [86]:
# Populate ligand dictionary.

liganddict["E20_ideal"] = {'id':"E20_ideal", 'chemname': "1-BENZYL-4-[(5,6-DIMETHOXY-1-INDANON-2-YL)METHYL]PIPERIDINE", 'sourcefile': "https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/00118E6700000001/SDF?response_type=save&response_basename=Conformer3D_COMPOUND_CID_1150567", 'source':"PubChem"}
liganddict["LS5_ideal"] = {'id':"LS5_ideal", 'chemname': "3-{[4-([AMINO(IMINO)METHYL]AMINOSULFONYL)ANILINO]METHYLENE}-2-OXO-2,3-DIHYDRO-1H-INDOLE", 'sourcefile': "https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/004450CC00000001/SDF?response_type=save&response_basename=Conformer3D_COMPOUND_CID_4477132", 'source':"PubChem"}
liganddict["VX6_ideal"] = {'id':"VX6_ideal", 'chemname': "CYCLOPROPANECARBOXYLIC ACID {4-[4-(4-METHYL-PIPERAZIN-1-YL)-6-(5-METHYL-2H-PYRAZOL-3-YLAMINO)-PYRIMIDIN-2-YLSULFANYL]-PHENYL}-AMIDE", 'sourcefile': "https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/0053D6B100000001/SDF?response_type=save&response_basename=Conformer3D_COMPOUND_CID_5494449", 'source':"PubChem"}
#liganddict["685_ideal"] = {'id':"685_ideal", 'chemname': "2-{2-[(3,5-dimethylphenyl)amino]pyrimidin-4-yl}-N-[(1S)-2-hydroxy-1-methylethyl]-4-methyl-1,3-thiazole-5-carboxamide", 'sourcefile': "https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/017F325400000001/SDF?response_type=save&response_basename=Conformer3D_COMPOUND_CID_25113172", 'source':"PubChem"}
#liganddict["0KP_ideal"] = {'id':"0KP_ideal", 'chemname': "(2E,5R)-2-imino-3-methyl-5-phenyl-5-[3-(pyridin-3-yl)phenyl]imidazolidin-4-one", 'sourcefile': "https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/0436923400000001/SDF?response_type=save&response_basename=Conformer3D_COMPOUND_CID_70685236", 'source':"PubChem"}
#liganddict["V50_ideal"] = {'id':"V50_ideal", 'chemname': "4-[(4,6-dimethylpyrimidin-2-yl)thio]-2,3,5,6-tetrafluorobenzenesulfonamide", 'sourcefile': "https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/043FF10800000002/SDF?response_type=save&response_basename=Conformer3D_COMPOUND_CID_71299336", 'source':"PubChem"}
#liganddict["2R6_ideal"] = {'id':"2R6_ideal", 'chemname': "8-(2-methylpropyl)-6-(3,6,6-trimethyl-4-oxo-4,5,6,7-tetrahydro-1H-indol-1-yl)-3,4-dihydroisoquinolin-1(2H)-one", 'sourcefile': "https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/046091AF00000001/SDF?response_type=save&response_basename=Conformer3D_COMPOUND_CID_73437615", 'source':"PubChem"}
liganddict["TSN_ideal"] = {'id':"TSN_ideal", 'chemname': "TRICHOSTATIN A", 'sourcefile': "https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/0006C93C00000002/SDF?response_type=save&response_basename=Conformer3D_COMPOUND_CID_444732", 'source':"PubChem"}
#liganddict["9XT_ideal"] = {'id':"9XT_ideal", 'chemname': "3-[(4-azanyl-3-methyl-phenyl)methyl]-7-(furan-2-yl)-[1,2,3]triazolo[4,5-d]pyrimidin-5-amine", 'sourcefile': "https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/014DC77D00000002/SDF?response_type=save&response_basename=Conformer3D_COMPOUND_CID_21874557", 'source':"PubChem"}
#liganddict["CAU_ideal"] = {'id':"CAU_ideal", 'chemname': "(2S)-1-(9H-Carbazol-4-yloxy)-3-(isopropylamino)propan-2-ol", 'sourcefile': "https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/00C6B86400000003/SDF?response_type=save&response_basename=Conformer3D_COMPOUND_CID_13023332", 'source':"PubChem"}
          

In [87]:
# Download all ligand files
import os

for key, value in liganddict.items():
    localfilename = value['id'] + "_" + value['source'] + ".sdf"
    value["localfilename"] = localfilename
    value["prep"] = "Wget 3d .sdf from PubChem"
    path = ligdowndir + localfilename
    
    if not os.path.exists(path):
        !wget -nv -O "{path}" "{value['sourcefile']}"
    else:
        print("File " + localfilename + " already downloaded. No wget necessary.")

!cp {ligdowndir}/* {ligdir}

wget: unable to resolve host address ‘{value['sourcefile']}’
File VX6_ideal_PubChem.sdf already downloaded. No wget necessary.
File LS5_ideal_PubChem.sdf already downloaded. No wget necessary.
File V50_ideal_PubChem.sdf already downloaded. No wget necessary.
File 2R6_ideal_PubChem.sdf already downloaded. No wget necessary.
File E20_ideal_PubChem.sdf already downloaded. No wget necessary.
File TSN_ideal_PubChem.sdf already downloaded. No wget necessary.


## Create Ligands from SMILES Chains

In [ ]:
ligand = {}
ligand["code"] = "E20"
ligand["name"] = "E20 created from SMILES"
ligand["smile"] = "COc1cc2c(cc1OC)C(=O)[C@@H](C2)CC3CCN(CC3)Cc4ccccc4"
ligand["source"] = "https://www.ebi.ac.uk/pdbe-srv/pdbechem/chemicalCompound/show/E20#structures-section"
ligand["filename"] = "E20.sdf"
ligand["prep"] = "obabel --gen3d"
liganddict["smilesE20"] = ligand

In [ ]:
# Prepare one or many ligands with Openbabel.
# Loop over keys and values
for key, value in liganddict.items():
    smilessdf = chemfilesdir + value["filename"]
    !obabel -:"{value["smile"]}" --gen3d -osdf -O"{smilessdf}"

#### Add SMILES preparation step to list of preparations ####

In [ ]:
prep = {}
prep["name"] = "obabel --gen3d"
prep["description"] = "OpenBabel executable used to convert SMILES chain into 3D .sdf file via '--gen3d' flag."
prep["functionname"] = "TBD"
prepdict["obabel --gen3d"] = prep

## Populate protein dictionary and then download files from protein database websites ##

In [ ]:
proteindict = {}

In [82]:
proteindict ["6o4w_Redo"] = {"id":"6o4w_Redo","name":"Acetylcholinesterase","keepchain":"B","keephet":"E20","source":"https://pdb-redo.eu/db/6o4w/6o4w_final.pdb"}
proteindict ["1KE9_RCBS"] = {"id":"1KE9_RCBS","name":"Cyclin-dependent kinase 2","keepchain":"A","keephet":"LS5","source":"https://files.rcsb.org/download/1KE9.pdb"}
proteindict ["2XYN_RCBS"] = {"id":"2XYN_RCBS","name":"Tyrosine-protein kinase ABL2","keepchain":"C","keephet":"VX6","keepsequence":"547","source":"https://files.rcsb.org/download/2XYN.pdb"}
#proteindict ["3emg_Redo"] = {"id":"3emg_Redo","name":"SYK kinase","keepchain":"A","keephet":"685","source":"https://pdb-redo.eu/db/3emg/3emg_final.pdb"}
#proteindict ["4djw_Redo"] = {"id":"4djw_Redo","name":"Beta-secretase 1","keepchain":"B","keephet":"0KP","source":"https://pdb-redo.eu/db/4djw/4djw_final.pdb"}
#proteindict ["4ht2_Redo"] = {"id":"4ht2_Redo","name":"Carbonic anhydrase II","keepchain":"A","keephet":"V50","source":"https://pdb-redo.eu/db/4ht2/4ht2_final.pdb"}
#proteindict ["4o09_Redo"] = {"id":"4o09_Redo","name":"HSP90 alpha","keepchain":"A","keephet":"2R6","source":"https://pdb-redo.eu/db/4o09/4o09_final.pdb"}
proteindict ["5edu_Redo"] = {"id":"5edu_Redo","name":"HDAC6","keepchain":"B","keephet":"TSN","source":"https://pdb-redo.eu/db/5edu/5edu_final.pdb"}
#proteindict ["5olh_Redo"] = {"id":"5olh_Redo","name":"Adenosine A2a receptor","keepchain":"A","keephet":"9XT","source":"https://pdb-redo.eu/db/5olh/5olh_final.pdb"}
#proteindict ["7BVQ_RCBS"] = {"id":"7BVQ_RCBS","name":"Dopamine D3 receptor","keepchain":"B","keephet":"CAU","source":"https://files.rcsb.org/download/7BVQ.pdb"}

In [ ]:
proteindict ["1KE9_RCBS_1"] = {"id":"1KE9_RCBS_1","name":"Cyclin-dependent kinase 2","keepchain":"A","keephet":"LS5","source":"https://files.rcsb.org/download/1KE9.pdb"}
proteindict ["1KE9_RCBS_2"] = {"id":"1KE9_RCBS_2","name":"Cyclin-dependent kinase 2","keepchain":"A","keephet":"LS5","source":"https://files.rcsb.org/download/1KE9.pdb"}
proteindict ["1KE9_RCBS_3"] = {"id":"1KE9_RCBS_3","name":"Cyclin-dependent kinase 2","keepchain":"A","keephet":"LS5","source":"https://files.rcsb.org/download/1KE9.pdb"}
proteindict ["1KE9_RCBS_4"] = {"id":"1KE9_RCBS_4","name":"Cyclin-dependent kinase 2","keepchain":"A","keephet":"LS5","source":"https://files.rcsb.org/download/1KE9.pdb"}
proteindict ["1KE9_RCBS_5"] = {"id":"1KE9_RCBS_5","name":"Cyclin-dependent kinase 2","keepchain":"A","keephet":"LS5","source":"https://files.rcsb.org/download/1KE9.pdb"}
proteindict ["2XYN_RCBS_1"] = {"id":"2XYN_RCBS_1","name":"Tyrosine-protein kinase ABL2","keepchain":"C","keephet":"VX6","source":"https://files.rcsb.org/download/2XYN.pdb"}
proteindict ["2XYN_RCBS_2"] = {"id":"2XYN_RCBS_2","name":"Tyrosine-protein kinase ABL2","keepchain":"C","keephet":"VX6","source":"https://files.rcsb.org/download/2XYN.pdb"}
proteindict ["2XYN_RCBS_3"] = {"id":"2XYN_RCBS_3","name":"Tyrosine-protein kinase ABL2","keepchain":"C","keephet":"VX6","source":"https://files.rcsb.org/download/2XYN.pdb"}
proteindict ["2XYN_RCBS_4"] = {"id":"2XYN_RCBS_4","name":"Tyrosine-protein kinase ABL2","keepchain":"C","keephet":"VX6","source":"https://files.rcsb.org/download/2XYN.pdb"}
proteindict ["2XYN_RCBS_5"] = {"id":"2XYN_RCBS_5","name":"Tyrosine-protein kinase ABL2","keepchain":"C","keephet":"VX6","source":"https://files.rcsb.org/download/2XYN.pdb"}


In [83]:
# Return just the filename (with extension) from a path.
import os
def get_filename(path):
    return os.path.basename(path)

In [84]:
# Loop over keys and values
for key, value in proteindict.items():
    filename = get_filename(value['source'])
    #localfilename = chemfilesdir + filename
    path = prodowndir + filename
    #!wget -nv -O "{localfilename}" "{value['source']}"
    value["localfilename"] = filename  


    if not os.path.exists(path):
        !wget -nv -O "{path}" "{value['source']}"
    else:
        print("File " + filename + " already downloaded. No wget necessary.")

File 2XYN.pdb already downloaded. No wget necessary.
File 6o4w_final.pdb already downloaded. No wget necessary.
File 1KE9.pdb already downloaded. No wget necessary.
File 5edu_final.pdb already downloaded. No wget necessary.


In [85]:
#print(proteindict)
lines = reportdict(proteindict, ["id","name","keephet","keepchain","source","localfilename"])
text = "\n".join(lines) + "\n"
print(text)

id         name                          keephet  keepchain  source                                      localfilename 
2XYN_RCBS  Tyrosine-protein kinase ABL2  VX6      C          https://files.rcsb.org/download/2XYN.pdb    2XYN.pdb      
6o4w_Redo  Acetylcholinesterase          E20      B          https://pdb-redo.eu/db/6o4w/6o4w_final.pdb  6o4w_final.pdb
1KE9_RCBS  Cyclin-dependent kinase 2     LS5      A          https://files.rcsb.org/download/1KE9.pdb    1KE9.pdb      
5edu_Redo  HDAC6                         TSN      B          https://pdb-redo.eu/db/5edu/5edu_final.pdb  5edu_final.pdb



## Examine proteins to determine what chains and ligands are present in the proteins. (Optional informational step) ##

In [ ]:
import gemmi

for key, value in proteindict.items():
    
    structure = gemmi.read_structure(prodowndir + value['localfilename'])
    #structure = gemmi.read_structure(chemfilesdir + "6O4w_rcbs.pdb")
    ligands = []
    
    for model in structure:
        for chain in model:
            for res in chain:
                if res.het_flag != ' ':  # hetero-residue
                    if res.name not in ("HOH", "WAT", "H2O"):
                        #if res.seqid.num == 604:
                        ligands.append((res.name, chain.name, res.seqid.num))

    print("protein: " + value['localfilename'])
    print(set(ligands))

In [ ]:
from Bio.PDB import PDBParser

for key, value in proteindict.items():
    
  parser = PDBParser(QUIET=True)
  structure = parser.get_structure("prot", prodowndir + value['localfilename'])
  #structure = parser.get_structure("prot", prodir + value['localfilename_fixed'])
    
  print("Protein: " + value['localfilename'])
    
  for model in structure:
    print(f"  Model {model.id}:")
    chain_ids = [chain.id for chain in model]
    print("    Chains:", ", ".join(chain_ids))



## Remove unneccessary chains ##

In [ ]:
#Chainstripper
from Bio.PDB import PDBParser, PDBIO, Select

def ChainStripper(protein):
    parser = PDBParser(QUIET=True)
    localfilename = protein["localfilename"]
    structure = parser.get_structure("struct", prodir + localfilename )
    io = PDBIO()
    io.set_structure(structure)
    filenameparts = localfilename.split(".")
    newfilename = filenameparts[0] + "_" + protein["keepchain"] + "." + filenameparts[1]
    io.save(prodir  + newfilename, KeepChain(protein["keepchain"]))
    protein['localfilename_chain'] = newfilename       
    print ("ChainStripper: " + protein["name"] + " => " + newfilename)
    
def KeepChain(chain_id):
    class KeepNon(Select):
        def accept_chain(self, chain):
            return chain.id == chain_id
    return KeepNon()

for key, value in proteindict.items():
    ChainStripper(value)
    

In [ ]:
#print(proteindict)
lines = reportdict(proteindict, ["id","name","keephet","keepchain","source","localfilename"])
text = "\n".join(lines) + "\n"
print(text)

## Extract the bound ligand

In [75]:
# Extract ligand based on ligand code, chain id, and optional sequence number.

from Bio.PDB import PDBParser, PDBIO, Select

class LigandSelect(Select):
    def __init__(self, ligand_name, chain_id=None, sequence=None, waters=None):
        self.ligand_name = ligand_name.upper()
        self.chain_id = chain_id  # e.g. "A", or None for any chain
        self.sequence_id = sequence  # e.g. "547", or None for any sequence
        self.waters = waters or {"HOH", "WAT", "H2O"}

    def accept_residue(self, residue):
        hetflag, resseq, icode = residue.id
        resname = residue.get_resname().strip().upper()
        chain = residue.get_parent()
        chain_id = chain.id

        # If a chain is specified, skip residues from other chains
        if self.chain_id is not None and chain_id != self.chain_id:
            return False

        # If a sequence is specified, skip all other sequences
        if self.sequence_id is not None and resseq != self.sequence_id:
            return False
        
        # Hetero residue with the desired name, not water
        #if hetflag != " " and resname == self.ligand_name and resname not in self.waters:
        if residue.id[0].startswith("H_") and hetflag != " " and resname == self.ligand_name and resname not in self.waters:         
            return True
        return False

    def accept_atom(self, atom):
        # Keep all atoms of accepted residues
        return True

def extract_ligand_by_resname(in_pdb, out_pdb, ligand_code, chain_id, seq_id):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("struct", in_pdb)
    io = PDBIO()
    io.set_structure(structure)
    io.save(out_pdb, LigandSelect(ligand_code, chain_id, seq_id))

In [88]:
from rdkit import Chem

for key, value in proteindict.items():
    
    proteinfilename = value["localfilename"] 
    filenameparts = proteinfilename.split(".")
    ligandfilename = value["keephet"] + "_redock_" + filenameparts[0] + "_" + value["keepchain"] + ".sdf"

    keepsequence = None
    if 'keepsequence' in value:
      keepsequence = int(value["keepsequence"])
    
    extract_ligand_by_resname(
        in_pdb = prodowndir + proteinfilename,
        out_pdb = ligdir + ligandfilename,
        ligand_code = value["keephet"], 
        chain_id = value["keepchain"],
        seq_id = keepsequence,
    ) 
    
    mol = Chem.MolFromPDBFile(ligdir + ligandfilename, removeHs=False) #removeHS=False preserves all hydrogens
    w = Chem.SDWriter(ligdir + ligandfilename)
    w.write(mol)
    w.close()

    ligandid = value["keephet"] + "_redock"
    ligand = {}
    ligand["id"] = ligandid
    ligand["name"] = value["keephet"] + " extracted from protein " + value["id"]
    ligand["localfilename"] = ligandfilename
    ligand["source"] = proteinfilename
    ligand["prep"] = "rdkit_save_H"
    liganddict[ligandid] = ligand
    
    print(proteinfilename + " -> " + ligandfilename)

2XYN.pdb -> VX6_redock_2XYN_C.sdf
6o4w_final.pdb -> E20_redock_6o4w_final_B.sdf
1KE9.pdb -> LS5_redock_1KE9_A.sdf
5edu_final.pdb -> TSN_redock_5edu_final_B.sdf


In [89]:
lines = reportdict(liganddict, ["id","source","prep","localfilename"])
print("\n".join(lines))

id          source          prep                       localfilename              
VX6_redock  2XYN.pdb        rdkit_save_H               VX6_redock_2XYN_C.sdf      
VX6_ideal   PubChem         Wget 3d .sdf from PubChem  VX6_ideal_PubChem.sdf      
LS5_ideal   PubChem         Wget 3d .sdf from PubChem  LS5_ideal_PubChem.sdf      
V50_ideal   PubChem         Wget 3d .sdf from PubChem  V50_ideal_PubChem.sdf      
2R6_ideal   PubChem         Wget 3d .sdf from PubChem  2R6_ideal_PubChem.sdf      
E20_ideal   PubChem         Wget 3d .sdf from PubChem  E20_ideal_PubChem.sdf      
TSN_ideal   PubChem         Wget 3d .sdf from PubChem  TSN_ideal_PubChem.sdf      
E20_redock  6o4w_final.pdb  rdkit_save_H               E20_redock_6o4w_final_B.sdf
LS5_redock  1KE9.pdb        rdkit_save_H               LS5_redock_1KE9_A.sdf      
TSN_redock  5edu_final.pdb  rdkit_save_H               TSN_redock_5edu_final_B.sdf


## Strip ligand with Obabel

In [ ]:
from openbabel import openbabel, pybel

def extract_ligand_by_resname(pdb_file, resname3, out_sdf, keepchain):
    mol = next(pybel.readfile("pdb", pdb_file))
    obmol = mol.OBMol
    ligand_atoms = []

    for res in openbabel.OBResidueIter(obmol):
        if res.GetName().strip() == resname3:
            print("chain:")
            print(res.GetChainNum())
            if (res.GetChainNum() == keepchain):
                for atom in openbabel.OBResidueAtomIter(res):
                    ligand_atoms.append(atom.GetIdx())

    if not ligand_atoms:
        raise ValueError(f"No residue named {resname3} found")

    # Create new molecule
    lig = openbabel.OBMol()
    atom_map = {}
    
    # Copy atoms
    for idx in ligand_atoms:
        old_atom = obmol.GetAtom(idx)
        new_atom = lig.NewAtom()
        new_atom.SetAtomicNum(old_atom.GetAtomicNum())
        new_atom.SetVector(old_atom.GetX(), old_atom.GetY(), old_atom.GetZ())
        atom_map[idx] = new_atom.GetIdx()
    
    # FIXED: Copy bonds properly
    for bond in openbabel.OBMolBondIter(obmol):
        begin_idx = bond.GetBeginAtomIdx()
        end_idx = bond.GetEndAtomIdx()
        if begin_idx in atom_map and end_idx in atom_map:
            lig.AddBond(atom_map[begin_idx], atom_map[end_idx], bond.GetBondOrder())
    
    # Write SDF
    lig_pb = pybel.Molecule(lig)
    lig_pb.write("sdf", out_sdf, overwrite=True)
    print(f"Ligand {resname3} written to {out_sdf}")

#!obabel "chemfiles/protein/4o09_final.pdb" -osdf -O ligands.sdf

In [ ]:
extract_ligand_by_resname("chemfiles/proteindownload/2XYN.pdb", "VX6", "chemfiles/ligand/VX6_obabel_C.sdf", "3")

In [ ]:
for key, value in proteindict.items():
    
    proteinfilename = value["localfilename_chain"] 
    filenameparts = proteinfilename.split(".")
    ligandfilename = value["keephet"] + "_redock_" + filenameparts[0] + "_obabel.sdf"

    print("Obabel ligand strip: " + proteinfilename + "=>" + ligandfilename)

    #extract_ligand_by_resname("chemfiles/protein/4o09_final.pdb", "2R6", "chemfiles/ligand/2R6.sdf")
    extract_ligand_by_resname(prodir + proteinfilename, value["keephet"], ligdir + ligandfilename)

    #append a new row to ligandict
    ligandid = value["keephet"] + "_redock_obabel"
    ligand = {}
    ligand["id"] = ligandid
    ligand["name"] = value["keephet"] + " extracted from protein " + value["id"]
    ligand["localfilename"] = ligandfilename
    ligand["source"] = proteinfilename
    ligand["prep"] = "obabel"
    liganddict[ligandid] = ligand
    


In [ ]:
lines = reportdict(liganddict, ["id","source","prep","localfilename"])
print("\n".join(lines))

#### Add ligand preparation options to list of preparations

In [ ]:
from pymol import cmd
import matplotlib.pyplot as plt
from IPython.display import Image

## Add preprocessed ligands to Dictionary

In [ ]:
liganddict["2R6_maya"] = {'id':"2R6_maya", 'chemname': "8-(2-methylpropyl)-6-(3,6,6-trimethyl-4-oxo-4,5,6,7-tetrahydro-1H-indol-1-yl)-3,4-dihydroisoquinolin-1(2H)-one", 'sourcefile': "https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/046091AF00000001/SDF?response_type=save&response_basename=Conformer3D_COMPOUND_CID_73437615", 'source':"PubChem"}
liganddict["TSN_maya"] = {'id':"TSN_maya", 'chemname': "TRICHOSTATIN A", 'sourcefile': "https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/0006C93C00000002/SDF?response_type=save&response_basename=Conformer3D_COMPOUND_CID_444732", 'source':"PubChem"}

ligandid = "2R6_maya"
ligand = liganddict[ligandid]
ligand["name"] = "2R6 extracted from protein by Maya"
ligand["localfilename"] = "4o09_final_ligand_2R6_A.pdb"
ligand["prep"] = "download"
liganddict[ligandid] = ligand

ligandid = "TSN_maya"
ligand = liganddict[ligandid]
ligand["name"] = "TSN extracted from protein by Maya"
ligand["localfilename"] = "5edu_final_ligand_TSN_B.pdb"
ligand["prep"] = "download"
liganddict[ligandid] = ligand

ligandid = "2R6_Ideal"
ligand = {}
ligand["name"] = "2R6 ideal supplied by Maya"
ligand["localfilename"] = "2R6_Ideal.sdf"
ligand["prep"] = "download"
liganddict[ligandid] = ligand

ligandid = "TSN_Ideal"
ligand = {}
ligand["name"] = "TSN ideal supplied by Maya"
ligand["localfilename"] = "TSN_Ideal.sdf"
ligand["prep"] = "download"
liganddict[ligandid] = ligand


In [ ]:
lines = reportdict(liganddict, ["id","source","prep","localfilename"])
print("\n".join(lines))

## Visualize Ligands

In [ ]:
from rdkit import Chem

ligandfile = ligdir + '2R6_ideal_PubChem.sdf'
ligandfileout = ligdir + '2R6_ideal_PubChem_NOH.sdf'

# Load SDF file (remove Hs on read - most efficient)
mol = Chem.MolFromMolFile(ligandfile, removeHs=True)

# Or if already loaded with Hs:
# mol = Chem.MolFromMolFile("ligand.sdf", removeHs=False)
# mol = Chem.RemoveHs(mol)

# Write H-free SDF
writer = Chem.SDWriter(ligandfileout)
writer.write(mol)
writer.close()

print(f"Atoms before: {Chem.MolFromMolFile(ligandfileout, removeHs=False).GetNumAtoms()}")
print(f"Atoms after:  {mol.GetNumAtoms()}")

In [ ]:
import nglview as nv
from rdkit import Chem

# From SDF
view = nv.show_structure_file(ligdir + '2R6_ideal_PubChem.sdf')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view = nv.show_structure_file(ligdir + '4o09_final_ligand_2R6_A.pdb')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view = nv.show_structure_file(ligdir + '2R6_redock_4o09_final_A_obabel.sdf')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view.display(gui=True) 

## Add ligand prep steps to list

In [51]:
prep = {}
prep["name"] = "rdkit_save_H"
prep["description"] = "Python 'Bio.PDB' package used to remove unnecessary chain(s) from the protein file."
prep["description"] += " Python 'Bio.PDB' package used to extract the ligand via code name."
prep["description"] += " Python 'rdkit' package used to convert the ligand file from PDB file to SDF file format."
prep["functionname"] = "TBD"
prepdict["rdkit_save_H"] = prep

In [ ]:
prep = {}
prep["name"] = "obabel"
prep["description"] = "Python 'Bio.PDB' package used to remove unnecessary chain(s) from the protein file."
prep["description"] += " Obabel Python module used to extract ligand from .pdb using three digit code."
prep["functionname"] = "TBD"
prepdict["obabel"] = prep

## Fix the protein

In [52]:
#PDBFixer python package is required by the following block. 
#This code expects that PDBFixer package is installed in the Conda environment that Jupyter is running within.
from pdbfixer import PDBFixer
from openmm.app import PDBFile
import numpy as np

In [53]:
def PDBfixfunc74(localprotein, savelocation, keepchain):
    fixer = PDBFixer(filename = localprotein)

    # List chains and their indices
    chains = list(fixer.topology.chains())
    trimchains = []
    trimchainids = []
    for i, chain in enumerate(chains):
      if (chain.id != keepchain):
        trimchains.append(i)
        trimchainids.append(chain.id)
    
    #fixer.removeChains(chainIndices=trimchains)
    fixer.removeChains(chainIds=trimchainids) 
    
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.removeHeterogens(False) #removes the complexed ligand. My original
    fixer.findMissingAtoms() # required by addMissingAtoms
    fixer.addMissingAtoms() # required by addMissingHydrogens
    #fixer.addMissingAtoms(seed=26) # addMissingAtoms() is required by addMissingHydrogens. The 'seed' parameter only affects heavy atoms and only between calls on the same machine.
    fixer.addMissingHydrogens(7.4)
    PDBFile.writeFile(fixer.topology, fixer.positions, open(savelocation, 'w'), keepIds=True)


In [ ]:
# PDBfixfunc74(prodowndir + "1KE9.pdb", "chemfiles/proteincomparison/dwaine_1KE9.pdb", "A")

In [54]:
prep = {}
prep["name"] = "PDBfixfunc74chain"
prep["description"] = "Python 'Bio.PDB' package Chainstripper() function used to remove extra chains."
prep["description"] += " PDBFixer package used to fix the protein with following options:"
prep["description"] += " fixer.removeChains(), findMissingResidues(), findNonstandardResidues(), replaceNonstandardResidues(),"
prep["description"] += " fixer.removeHeterogens(False), findMissingAtoms(), addMissingAtoms(), addMissingHydrogens(7.4)"
prep["functionname"] = "PDBfixfunc74()"
prepdict["PDBfixfunc"] = prep

In [90]:
for key, value in proteindict.items():

    localfilename = value["localfilename"]
    keepchain = value["keepchain"]
    filenameparts = localfilename.split(".")
    #localfilename_fixed = filenameparts[0] + "_" + keepchain + "_fixed." + filenameparts[1]
    localfilename_fixed = key + "_" + keepchain + "_fixed." + filenameparts[1]
    
    PDBfixfunc74(prodowndir + localfilename, prodir + localfilename_fixed, keepchain)
    
    value["localfilename_fixed"] = localfilename_fixed
    value["prep"] = "PDBfixfunc74"
    print("PDBfixfunc74() " + localfilename + " => " + localfilename_fixed)

PDBfixfunc74() 2XYN.pdb => 2XYN_RCBS_C_fixed.pdb
PDBfixfunc74() 6o4w_final.pdb => 6o4w_Redo_B_fixed.pdb
PDBfixfunc74() 1KE9.pdb => 1KE9_RCBS_A_fixed.pdb
PDBfixfunc74() 5edu_final.pdb => 5edu_Redo_B_fixed.pdb


In [91]:
lines = reportdict(proteindict, ["id","name","prep","localfilename_fixed"])
print("\n".join(lines))

id         name                          prep          localfilename_fixed  
2XYN_RCBS  Tyrosine-protein kinase ABL2  PDBfixfunc74  2XYN_RCBS_C_fixed.pdb
6o4w_Redo  Acetylcholinesterase          PDBfixfunc74  6o4w_Redo_B_fixed.pdb
1KE9_RCBS  Cyclin-dependent kinase 2     PDBfixfunc74  1KE9_RCBS_A_fixed.pdb
5edu_Redo  HDAC6                         PDBfixfunc74  5edu_Redo_B_fixed.pdb


## Add preprocessed proteins to Dictionary

In [ ]:
proteinid = "4o09_maya"
protein = {}
protein["name"] = "4o09_final processed and supplied by Maya"
protein["localfilename_fixed"] = "4o09_final_Fixed_ChainA.pdb"
proteindict[proteinid] = protein

proteinid = "5edu_maya"
protein = {}
protein["name"] = "5edu_final processed and supplied by Maya"
protein["localfilename_fixed"] = "5edu_final_Fixed_ChainB.pdb"
proteindict[proteinid] = protein

In [ ]:
lines = reportdict(proteindict, ["id","name","prep","localfilename_fixed"])
print("\n".join(lines))

## Setup scoring function and results structure

In [57]:
# Get values from line 1 of results section from GNINA results file.
def get_mode1_values(path):
    mode_section = False
    with open(path) as f:
        mode_section = False
        for line in f:
            line = line.strip()
            if line.startswith("mode"):
                mode_section = True
            if not line or line.startswith("mode") or line.startswith("-") or line.startswith("|") or mode_section is False:
                continue
            parts = line.split()
            mode = int(parts[0])
            if mode == 1:
                affinity = float(parts[1])
                intramol = float(parts[2])
                cnn_pose = float(parts[3])
                cnn_aff = float(parts[4])
                return mode, affinity, intramol, cnn_pose, cnn_aff
# Example usage:   mode, affinity, intramol, cnn_pose, cnn_aff = get_mode1_values("workflow03/log.txt")

In [58]:
gninaoptdict = {}

gninaopt = {}
gninaopt["id"] = "gninaopt1"
gninaopt["opt"] = "--exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu" 
gninaoptdict["gninaopt1"] = gninaopt

gninaopt = {}
gninaopt["id"] = "gninaopt2"
gninaopt["opt"] = "--exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu" 
gninaoptdict["gninaopt2"] = gninaopt

In [59]:
print (gninaoptdict)
cols = ["id","opt"]
print(reportdict(gninaoptdict,cols))

{'gninaopt1': {'id': 'gninaopt1', 'opt': '--exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu'}, 'gninaopt2': {'id': 'gninaopt2', 'opt': '--exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu'}}
['id         opt                                                                              ', 'gninaopt1  --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu', 'gninaopt2  --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu   ']


## GNINA and reporting functions

In [ ]:
#Examples
#output = !~/gnina-binary/gnina -r proteins_pdbfixtest/2ama_fixer_andy_minusH.pdb -l proteins_pdbfixtest/Trenbolone.sdf --autobox_ligand proteins_pdbfixtest/2ama-ligand.pdb -o docked.sdf.gz  --exhaustiveness=64 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --log proteins_pdbfixtest/log.txt
# Predict against complexed ligand.
#!~/octoberproject/gnina -r 8GUTciffixed4.pdb -l 8GUT-ligand.sdf --autobox_ligand 8GUT-ligand.sdf -o docked.sdf.gz  --exhaustiveness=64 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --log log.txt --no_gpu  

In [60]:
# Call Gnina, parse results from RMSD calculation, call function that parse results from both log files and appends to results variable

#pro = subdir + "6O4w_rcbs_chainB_fixed.pdb"
#lig = subdir + "6O4w_rcbs_chainB_E20.sdf"
#box = subdir + "6O4w_rcbs_chainB_E20.sdf"
#docked = subdir + "docked.sdf"
#log = subdir + "gninalog.txt"
#!~/octoberproject/gnina -r "{pro}" -l "{lig}" --autobox_ligand "{box}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu  

def rungnina(proteinid,ligandid,boxid):
    protein = proteindict[proteinid]
    ligand = liganddict[ligandid]
    box = liganddict[boxid]
    p = prodir + protein["localfilename_fixed"]
    l = ligdir + ligand["localfilename"]
    b = ligdir + box["localfilename"]
    #!~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu  
    !~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu  

    #Compute RMSD of two docked ligand files and save results to rmsdlog file.
    !obrms -f "{b}" "{docked}" | tee "{rmsdlog}"

    appendresults(proteinid,ligandid,boxid)

In [61]:
# Get values from text files, append to results data, print to screen.
# Note: The CNN_VS score is also report in the docked.sdf file.
import pandas as pd 

def appendresults(pro, lig, box):
    
    resultsdf = getdockedresults(docked)
    for index, row in resultsdf.iterrows():
        new_row = (pro + " + " + lig + " @ " + box , "{:.4f}".format(row['CNNscore']), "{:.4f}".format(row['CNN_VS']), "{:.4f}".format(row['RMSD']))
        data.append(new_row)

In [62]:
def getdockedresults(docked):
    from rdkit import Chem
    import pandas as pd  

    rmsddf = pd.read_csv(rmsdlog, sep=" ", header=None)
    
    rows = []
    for i, mol in enumerate(Chem.SDMolSupplier(docked)):
        if mol is None:
            continue
        rows.append({
            "pose": i,
            "CNNscore": float(mol.GetProp("CNNscore")),
            "CNN_VS": float(mol.GetProp("CNN_VS")),
            "CNNaffinity": float(mol.GetProp("CNNaffinity")),
            "RMSD": float(rmsddf.iloc[i,2]),
        })
    df = pd.DataFrame(rows)
    return(df)

In [63]:
def reporttable(rows):
    lines = []
    # rows: list of tuples, all same length
    if not rows:
        return
    # Number of columns
    n_cols = len(rows[0])
    # Max width per column (over all rows)
    widths = [
        max(len(str(row[col])) for row in rows)
        for col in range(n_cols)
    ]
    # Print each row with aligned columns
    for row in rows:
        line = "  ".join(
            f"{str(value):<{widths[i]}}"
            for i, value in enumerate(row)
        )
        #print(line)
        lines.append(line)

    return (lines)

In [64]:
#Report out results with details of Ligands and Proteins used
from datetime import datetime

now = datetime.now()
stamp = now.strftime("%Y-%m-%d %H:%M:%S")
unixtime = int(datetime.now().timestamp())

def reportall():
    lines = []

    lines.append("Report produced " + stamp + " experiment id: " + str(unixtime))

    lines.append("")
    lines.append(experimentdescription)

    lines.append("")
    cols = ["name","functionname","description"]
    lines.append("File preparation descriptions:")
    lines = lines + reportdict(prepdict,cols)
                
    lines.append("")
    cols = ["id", "name","prep","localfilename_fixed"]
    lines.append("Proteins:")
    lines = lines + reportdict(proteindict, cols)

    lines.append("")
    cols = ["id", "name","prep","localfilename"]
    lines.append("Ligands:")
    lines = lines + reportdict(liganddict, cols)

    lines.append("")
    cols = ["id", "opt"]
    lines.append("Gnina options:")
    lines = lines + reportdict(gninaoptdict,cols)

    lines.append("")
    lines.append("Results:")
    newrow = ("proteinid + ligandid @ ligandid", "CNN Score", "CNN_VS", "RMSD")     
    outputlist = list(data)
    outputlist.insert(0, newrow)
    lines = lines + reporttable(outputlist)

    return lines;

## Call Gnina

In [92]:
#Which ligands are available?
lines = reportdict(liganddict, ["id","source","prep","localfilename"])
print("\n".join(lines))

id          source          prep                       localfilename              
VX6_redock  2XYN.pdb        rdkit_save_H               VX6_redock_2XYN_C.sdf      
VX6_ideal   PubChem         Wget 3d .sdf from PubChem  VX6_ideal_PubChem.sdf      
LS5_ideal   PubChem         Wget 3d .sdf from PubChem  LS5_ideal_PubChem.sdf      
V50_ideal   PubChem         Wget 3d .sdf from PubChem  V50_ideal_PubChem.sdf      
2R6_ideal   PubChem         Wget 3d .sdf from PubChem  2R6_ideal_PubChem.sdf      
E20_ideal   PubChem         Wget 3d .sdf from PubChem  E20_ideal_PubChem.sdf      
TSN_ideal   PubChem         Wget 3d .sdf from PubChem  TSN_ideal_PubChem.sdf      
E20_redock  6o4w_final.pdb  rdkit_save_H               E20_redock_6o4w_final_B.sdf
LS5_redock  1KE9.pdb        rdkit_save_H               LS5_redock_1KE9_A.sdf      
TSN_redock  5edu_final.pdb  rdkit_save_H               TSN_redock_5edu_final_B.sdf


In [93]:
#Which proteins are available?
lines = reportdict(proteindict, ["id","name","prep","localfilename_fixed"])
print("\n".join(lines))

id         name                          prep          localfilename_fixed  
2XYN_RCBS  Tyrosine-protein kinase ABL2  PDBfixfunc74  2XYN_RCBS_C_fixed.pdb
6o4w_Redo  Acetylcholinesterase          PDBfixfunc74  6o4w_Redo_B_fixed.pdb
1KE9_RCBS  Cyclin-dependent kinase 2     PDBfixfunc74  1KE9_RCBS_A_fixed.pdb
5edu_Redo  HDAC6                         PDBfixfunc74  5edu_Redo_B_fixed.pdb


In [94]:
data = []

In [95]:
row = ["Buccheri_604W", "0.9200", "7.2900", "1.7100"]
data.append(row)
rungnina("6o4w_Redo", "E20_ideal", "E20_redock")

row = ["Buccheri_1KE9", "0.9700", "6.5200", "1.8000"]
data.append(row)
rungnina("1KE9_RCBS", "LS5_ideal", "LS5_redock")

row = ["Buccheri_2XYN", "0.9900", "8.4700", "0.7900"] 
data.append(row)
rungnina("2XYN_RCBS", "VX6_ideal", "VX6_redock")

row = ["Buccheri_5edu", "0.9700", "6.8800", "1.8000"]
data.append(row)
rungnina("5edu_Redo", "TSN_ideal", "TSN_redock")

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r chemfiles/protein/6o4w_Redo_B_fixed.pdb -l chemfiles/ligand/E20_ideal_PubChem.sdf --autobox_ligand chemfiles/ligand/E20_redock_6o4w_final_B.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
1150567 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score 

[10:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r chemfiles/protein/1KE9_RCBS_A_fixed.pdb -l chemfiles/ligand/LS5_ideal_PubChem.sdf --autobox_ligand chemfiles/ligand/LS5_redock_1KE9_A.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
4477132 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score | affi

[10:19:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:19:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:19:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:19:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:19:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:19:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:19:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:19:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:19:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r chemfiles/protein/2XYN_RCBS_C_fixed.pdb -l chemfiles/ligand/VX6_ideal_PubChem.sdf --autobox_ligand chemfiles/ligand/VX6_redock_2XYN_C.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
5494449 | pose 0 | initial pose not within box
5494449 | pose 0 | ligand outside box
5494449 | pose 0 | ligand outside box
5494449 | pose 0 | ligand o

[10:21:24] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:21:24] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:21:24] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:21:24] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:21:24] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:21:24] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:21:24] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:21:24] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:21:24] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r chemfiles/protein/5edu_Redo_B_fixed.pdb -l chemfiles/ligand/TSN_ideal_PubChem.sdf --autobox_ligand chemfiles/ligand/TSN_redock_5edu_final_B.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
444732 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score |

[10:22:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:22:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:22:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:22:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:22:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:22:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:22:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:22:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:22:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

## Display all results

In [96]:
#This block presents the results to stdout.
lines = reportall()
text = "\n".join(lines) + "\n"
print(text)

Report produced 2026-03-05 09:48:40 experiment id: 1772732920

Workflow 11 is Andy's 'Top 9' experiment extended. We will report results for top 9 poses but with multiple PDBFixer calls.

File preparation descriptions:
name               functionname    description                                                                                                                                                                                                                                                                                                                                         
rdkit_save_H  TBD             Python 'Bio.PDB' package used to remove unnecessary chain(s) from the protein file. Python 'Bio.PDB' package used to extract the ligand via code name. Python 'rdkit' package used to convert the ligand file from PDB file to SDF file format.                                                                                                     
PDBfixfunc    PDBfixfunc74()  Pyth

In [97]:
#This block will append results listed above to master results file.

with open(masterlog, "a") as f:
    f.write(text)

## Notes